<a href="https://colab.research.google.com/github/adytriaby/PCVK_Ganjil2026/blob/main/week6_PCVK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 7 — Filter Spasial
### Pengolahan Citra dan Visi Komputer

**Nama:** [isi nama]
**NIM:** [isi NIM]
**Kelas:** [isi kelas]

---

Notebook ini mengerjakan **Modul 6 – Filter Spasial**:
- **D. Praktikum** — fungsi konvolusi manual (tanpa `cv2.filter2D`), average, low pass, high pass, sharpen, emboss, edge detection, Gaussian 21x21.
- **E. Percobaan 1–7** — filter library, bilateral, guided filter, feature map CNN, beauty/vintage, kartun, malam, pagi + kabut.
- **Tugas Praktikum 1–3** — Mean vs Median pada salt & pepper, pemilihan kernel sharpening, dan filter kartun.

> **Catatan pengerjaan:** jalankan `Runtime > Restart and Run All` sebelum dikumpulkan.

## 0. Setup

Import library dan pengaturan tampilan. `google.colab` hanya tersedia saat dijalankan di Colab, jadi dibungkus `try/except` agar notebook tetap bisa diuji di lokal.

In [ ]:
import os
import math
import random

import cv2
import numpy as np
import matplotlib.pyplot as plt

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

plt.rcParams["figure.dpi"] = 110
plt.rcParams["image.cmap"] = "gray"
np.random.seed(42)
random.seed(42)

print("OpenCV :", cv2.__version__)
print("NumPy  :", np.__version__)
print("Colab  :", IN_COLAB)

### Upload citra

Foto tugas (`djawatan.jpg`) diupload manual. Jika nama file berbeda, notebook otomatis memakai file gambar pertama yang diupload.

In [ ]:
NAMA_FILE = "djawatan.jpg"

if not os.path.exists(NAMA_FILE):
    if IN_COLAB:
        print(f"File '{NAMA_FILE}' belum ada. Silakan upload foto tugas kamu:")
        uploaded = files.upload()
        if uploaded and NAMA_FILE not in uploaded:
            kandidat = [n for n in uploaded if n.lower().endswith((".jpg", ".jpeg", ".png"))]
            if not kandidat:
                raise FileNotFoundError("Tidak ada file gambar yang diupload.")
            os.rename(kandidat[0], NAMA_FILE)
    else:
        raise FileNotFoundError(
            f"Letakkan '{NAMA_FILE}' di folder yang sama dengan notebook ini."
        )

img_bgr = cv2.imread(NAMA_FILE)
if img_bgr is None:
    raise ValueError(f"'{NAMA_FILE}' tidak bisa dibaca sebagai citra.")

img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

print("Ukuran citra :", img_rgb.shape, "(tinggi, lebar, kanal)")
print("Tipe data    :", img_rgb.dtype)
print("Rentang nilai:", img_rgb.min(), "-", img_rgb.max())

### Fungsi bantu untuk menampilkan citra

In [ ]:
def tampilkan(daftar_citra, judul=None, cmap="gray", ncols=None, skala=4.2):
    """Menampilkan satu atau beberapa citra dalam grid."""
    if not isinstance(daftar_citra, (list, tuple)):
        daftar_citra = [daftar_citra]
    n = len(daftar_citra)
    if judul is None:
        judul = [""] * n
    ncols = ncols or n
    nrows = math.ceil(n / ncols)

    fig, axes = plt.subplots(nrows, ncols, figsize=(skala * ncols, skala * nrows))
    axes = np.atleast_1d(axes).ravel()

    for ax, im, t in zip(axes, daftar_citra, judul):
        ax.imshow(im, cmap=cmap if np.ndim(im) == 2 else None)
        ax.set_title(t, fontsize=10)
        ax.axis("off")

    for ax in axes[n:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


def tampil_berdampingan(kiri, kanan, judul_kiri="Asli", judul_kanan="Hasil", skala=6):
    tampilkan([kiri, kanan], [judul_kiri, judul_kanan], ncols=2, skala=skala)


def ke_uint8(citra):
    """Clip ke 0-255 lalu ubah ke uint8 (hasil konvolusi bisa negatif / >255)."""
    return np.clip(citra, 0, 255).astype(np.uint8)


tampilkan([img_rgb, img_gray], ["Citra asli (RGB)", "Citra keabuan (grayscale)"], ncols=2, skala=6)

---

# D. Praktikum Filter Spasial

## D.1 Fungsi konvolusi manual

Fungsi `konvolusi()` dibuat **tanpa** `cv2.filter2D`, `scipy`, atau metode konvolusi bawaan OpenCV. Implementasi memakai *sliding window* NumPy (vectorized) agar cepat, sedangkan `konvolusi_naive()` adalah versi literal algoritma modul (loop 4 tingkat) yang dipakai untuk membuktikan hasilnya sama.

**Penanganan border** (sesuai modul):
- `zero` → piksel pinggir diberi nilai 0 (zero padding)
- `replicate` → duplikasi piksel tepi
- `valid` → piksel pinggir tidak dikonvolusi (output lebih kecil)

**Catatan:** kernel tidak dibalik, mengikuti pseudocode modul dan perilaku `cv2.filter2D` (korelasi). Untuk kernel simetris (average, Gaussian, Laplacian) hasilnya identik dengan konvolusi matematis.

In [ ]:
def _padding_citra(citra, kh, kw, mode):
    """Tambahkan border: 'zero' (nilai 0) atau 'replicate' (duplikasi tepi)."""
    atas, bawah = kh // 2, kh - 1 - kh // 2
    kiri, kanan = kw // 2, kw - 1 - kw // 2
    if mode == "zero":
        return np.pad(citra, ((atas, bawah), (kiri, kanan)),
                      mode="constant", constant_values=0)
    return np.pad(citra, ((atas, bawah), (kiri, kanan)), mode="edge")


def konvolusi(citra, kernel, padding="zero"):
    """Konvolusi 2D manual (vectorized).

    Parameters
    ----------
    citra   : array 2D (grayscale) atau 3D (RGB)
    kernel  : array 2D berukuran kh x kw
    padding : 'zero' | 'replicate' | 'valid'

    Returns
    -------
    array float64 hasil konvolusi
    """
    img = np.asarray(citra, dtype=np.float64)
    ker = np.asarray(kernel, dtype=np.float64)

    if img.ndim == 3:
        return np.stack(
            [konvolusi(img[..., c], ker, padding) for c in range(img.shape[2])],
            axis=-1,
        )

    kh, kw = ker.shape
    if padding == "valid":
        jendela = np.lib.stride_tricks.sliding_window_view(img, (kh, kw))
        return np.einsum("ijkl,kl->ij", jendela, ker)

    if padding not in ("zero", "replicate"):
        raise ValueError("padding harus 'zero', 'replicate', atau 'valid'")

    padded = _padding_citra(img, kh, kw, padding)
    jendela = np.lib.stride_tricks.sliding_window_view(padded, (kh, kw))
    return np.einsum("ijkl,kl->ij", jendela, ker)


def konvolusi_naive(citra, kernel, padding="zero"):
    """Versi literal algoritma modul (loop x, y, k1, k2) untuk verifikasi."""
    img = np.asarray(citra, dtype=np.float64)
    ker = np.asarray(kernel, dtype=np.float64)
    kh, kw = ker.shape

    if padding == "valid":
        H, W = img.shape[0] - kh + 1, img.shape[1] - kw + 1
        padded = img
    else:
        H, W = img.shape
        padded = _padding_citra(img, kh, kw, padding)

    hasil = np.zeros((H, W), dtype=np.float64)
    for y in range(H):
        for x in range(W):
            total = 0.0
            for k1 in range(kh):
                for k2 in range(kw):
                    total += ker[k1, k2] * padded[y + k1, x + k2]
            hasil[y, x] = total
    return hasil

## D.2 Verifikasi fungsi konvolusi

Uji dengan **contoh perhitungan pada modul**: jendela 3x3 berisi `5 5 6 / 5 4 4 / 0 0 2` dikalikan kernel 1/4. Hasil di posisi (2,2) seharusnya **7,75**:

`(5+5+6+5+4+4+0+0+2) x 1/4 = 31 / 4 = 7,75`

Setelah itu bandingkan versi naive dengan versi vectorized pada ketiga mode padding.

In [ ]:
contoh_citra = np.zeros((5, 5), dtype=float)
contoh_citra[1:4, 1:4] = np.array([
    [5, 5, 6],
    [5, 4, 4],
    [0, 0, 2],
])
kernel_contoh = np.ones((3, 3)) / 4

nilai = konvolusi_naive(contoh_citra, kernel_contoh, padding="zero")[2, 2]
print(f"Nilai konvolusi di (2,2) : {nilai:.2f}")
print(f"Nilai pada modul         : 7.75")
print(f"Cocok                    : {np.isclose(nilai, 7.75)}")

# Contoh lain: matriks 5x5 pada modul dengan kernel average 1/9
matriks_modul = np.array([
    [4, 4, 3, 5, 4],
    [6, 6, 5, 5, 2],
    [5, 6, 6, 6, 2],
    [6, 7, 5, 5, 3],
    [3, 5, 2, 4, 4],
], dtype=float)
print(f"\nMatriks 5x5 modul, kernel 1/9, nilai (2,2): "
      f"{konvolusi_naive(matriks_modul, np.ones((3, 3)) / 9)[2, 2]:.2f}")

print("\nPerbandingan naive vs vectorized:")
uji = np.random.randint(0, 256, (8, 8)).astype(float)
kernel_uji = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=float)
for mode in ("zero", "replicate", "valid"):
    a = konvolusi_naive(uji, kernel_uji, mode)
    b = konvolusi(uji, kernel_uji, mode)
    print(f"  padding={mode:<10} shape={str(a.shape):<10} sama={np.allclose(a, b)}")

## D.3 Sharpening (contoh langkah modul)

Kernel penajaman 3x3 diterapkan pada citra keabuan. Nilai negatif di sekitar pusat memperkuat perbedaan intensitas sehingga tepi terlihat lebih tajam.

In [ ]:
kernel_sharpen = np.array([
    [0, -1, 0],
    [-1, 5, -1],
    [0, -1, 0],
], dtype=float)

hasil_sharpen = konvolusi(img_gray, kernel_sharpen, padding="replicate")

tampil_berdampingan(
    img_gray, ke_uint8(hasil_sharpen),
    "Asli (grayscale)", "Sharpen 3x3 (konvolusi manual)",
)

## D.4 Image filter: Average, Low Pass, High Pass, dan lainnya

Kernel yang dipakai:

| Filter | Kernel |
|---|---|
| Average 3x3 | semua elemen 1/9 |
| Low Pass (Gaussian 3x3) | `[1 2 1; 2 4 2; 1 2 1] / 16` |
| High Pass | pusat 8, sisanya -1 |
| Sharpen | pusat 5, silang -1 |
| Emboss | `[-2 -1 0; -1 1 1; 0 1 2]` |
| Left Sobel Edge | `[1 1 -1; 1 1 -1; 1 1 -1]` |
| Canny Edge (kernel modul) | pusat 8, sisanya -1 (pendekatan Laplacian) |
| Gaussian Blur 21x21 | dibangkitkan dengan `cv2.getGaussianKernel` |

> Kernel edge/high-pass menghasilkan nilai negatif. Hasil ditampilkan setelah di-*clip* ke 0–255, sehingga tepi muncul terang di atas latar gelap.

In [ ]:
KERNEL_FILTER = {
    "Average 3x3": np.ones((3, 3)) / 9,
    "Low Pass (Gaussian 3x3)": np.array([
        [1, 2, 1],
        [2, 4, 2],
        [1, 2, 1],
    ], dtype=float) / 16,
    "High Pass": np.array([
        [-1, -1, -1],
        [-1, 8, -1],
        [-1, -1, -1],
    ], dtype=float),
    "Sharpen": kernel_sharpen,
    "Emboss": np.array([
        [-2, -1, 0],
        [-1, 1, 1],
        [0, 1, 2],
    ], dtype=float),
    "Left Sobel Edge": np.array([
        [1, 1, -1],
        [1, 1, -1],
        [1, 1, -1],
    ], dtype=float),
    "Canny Edge (kernel modul)": np.array([
        [-1, -1, -1],
        [-1, 8, -1],
        [-1, -1, -1],
    ], dtype=float),
}

# Gaussian 21x21 (sesuai petunjuk modul)
ukuran_gauss = 21
sigma_gauss = math.sqrt(ukuran_gauss)
gauss_1d = cv2.getGaussianKernel(ukuran_gauss, sigma_gauss)
kernel_gauss21 = gauss_1d @ gauss_1d.T
KERNEL_FILTER["Gaussian Blur 21x21"] = kernel_gauss21

for nama, kernel in KERNEL_FILTER.items():
    print(f"{nama:<28} shape={kernel.shape}  jumlah={kernel.sum():.4f}")

In [ ]:
hasil_filter = {}
for nama, kernel in KERNEL_FILTER.items():
    hasil_filter[nama] = ke_uint8(konvolusi(img_gray, kernel, padding="replicate"))

tampilkan(
    [img_gray] + list(hasil_filter.values()),
    ["Asli (grayscale)"] + list(hasil_filter.keys()),
    ncols=3,
    skala=4.5,
)

### Analisis Praktikum D

1. **Average & Low Pass (Gaussian)** menurunkan frekuensi tinggi sehingga citra tampak halus dan noise menyebar. Gaussian 21x21 memberi efek blur paling kuat karena bobot piksel pusat mendominasi dan jangkauan kernel sangat lebar.
2. **High Pass, Sharpen, dan Canny (Laplacian)** menonjolkan perubahan intensitas. Area rata menjadi gelap, sedangkan tepi objek menjadi terang. Sharpen bekerja lebih "aman" karena mempertahankan komponen asli citra (jumlah kernel = 1).
3. **Emboss** memberi kesan relief tiga dimensi karena kernel asimetris: satu sisi negatif dan sisi lain positif, sehingga tepi terlihat seperti bayangan.
4. **Left Sobel** hanya merespons gradien pada satu arah, sehingga tepi yang tegak lurus arah tersebut lebih dominan.
5. **Border:** mode `replicate` dipakai pada percobaan ini agar tepi citra tidak menjadi garis gelap/terang palsu seperti pada `zero` padding. Mode `valid` menyusutkan output dan membiarkan piksel pinggir tidak diproses.

---

# E. Filter Library dan Filter Modern

## Percobaan 1 — Gaussian, Sharpen, dan Canny dengan `cv2.filter2D`

Berbeda dengan Praktikum D, di sini filtering dilakukan oleh library OpenCV pada citra **RGB**. Canny memakai pipeline gradient bawaan OpenCV (bukan kernel tunggal).

In [ ]:
kernel_gauss9 = cv2.getGaussianKernel(9, 2.0)
kernel_gauss9 = kernel_gauss9 @ kernel_gauss9.T

gaussian_lib = cv2.filter2D(img_rgb, -1, kernel_gauss9)
sharpen_lib = cv2.filter2D(img_rgb, -1, kernel_sharpen)
canny_lib = cv2.Canny(img_gray, 100, 200)

tampilkan(
    [img_rgb, gaussian_lib, sharpen_lib, canny_lib],
    ["Asli (RGB)", "Gaussian Blur (filter2D)", "Sharpen (filter2D)", "Canny Edge (library)"],
    ncols=2,
    skala=6,
)

## Percobaan 2 — Bilateral Filter dan Guided Filter

- **Bilateral filter** menghaluskan area datar tetapi mempertahankan tepi karena bobotnya mempertimbangkan jarak spasial *dan* perbedaan warna.
- **Guided filter** memodelkan output sebagai fungsi linear dari citra pemandu pada jendela lokal. Implementasi di bawah dibuat manual (box filter) karena `cv2.ximgproc` tidak tersedia di Google Colab.

In [ ]:
def _guided_2d(g, p, radius, eps):
    """Guided filter untuk satu kanal (input sudah dinormalisasi 0..1)."""
    ukuran = (2 * radius + 1, 2 * radius + 1)

    mean_g = cv2.boxFilter(g, -1, ukuran)
    mean_p = cv2.boxFilter(p, -1, ukuran)
    corr_gg = cv2.boxFilter(g * g, -1, ukuran)
    corr_gp = cv2.boxFilter(g * p, -1, ukuran)

    var_g = corr_gg - mean_g * mean_g
    cov_gp = corr_gp - mean_g * mean_p

    a = cov_gp / (var_g + eps)
    b = mean_p - a * mean_g

    mean_a = cv2.boxFilter(a, -1, ukuran)
    mean_b = cv2.boxFilter(b, -1, ukuran)
    return mean_a * g + mean_b


def guided_filter(guide, src, radius=8, eps=1e-2):
    """Guided filter manual (He et al.) untuk citra grayscale maupun RGB.

    Output bernilai 0..1 (float).
    """
    g = np.asarray(guide, dtype=np.float64) / 255.0
    p = np.asarray(src, dtype=np.float64) / 255.0

    if g.ndim == 3:
        return np.stack(
            [_guided_2d(g[..., c], p[..., c], radius, eps) for c in range(g.shape[2])],
            axis=-1,
        )
    return _guided_2d(g, p, radius, eps)


bilateral_lib = cv2.bilateralFilter(img_rgb, d=9, sigmaColor=75, sigmaSpace=75)
guided_lib = ke_uint8(guided_filter(img_rgb, img_rgb, radius=8, eps=1e-2) * 255)

tampilkan(
    [img_rgb, bilateral_lib, guided_lib],
    ["Asli (RGB)", "Bilateral Filter", "Guided Filter (manual)"],
    ncols=3,
    skala=5.5,
)

## Percobaan 3 — Feature Map pada CNN

Setiap kernel pada lapisan konvolusi CNN berfungsi seperti filter spasial yang "belajar" mendeteksi pola tertentu (tepi, tekstur, sudut). Di sini kernel dibuat **acak** untuk mensimulasikan feature map.

**Coba jalankan sel berikut beberapa kali**, lalu perhatikan kesimpulan di bawah.

In [ ]:
def feature_maps(citra, jumlah=6, ukuran=3):
    """Hasilkan feature map dari beberapa kernel acak."""
    hasil, judul, kernels = [], [], []
    for i in range(jumlah):
        kernel = np.random.randn(ukuran, ukuran)
        kernel /= np.abs(kernel).sum() + 1e-8
        fmap = konvolusi(citra.astype(float), kernel, padding="replicate")
        fmap = (fmap - fmap.min()) / (fmap.max() - fmap.min() + 1e-8)
        hasil.append(fmap)
        judul.append(f"Kernel acak #{i + 1}")
        kernels.append(kernel)
    return hasil, judul, kernels


fm, judul_fm, kernels_fm = feature_maps(img_gray, jumlah=6)
tampilkan([img_gray] + fm, ["Input (grayscale)"] + judul_fm, ncols=4, skala=4)

**Kesimpulan Percobaan 3:** setiap kernel acak menghasilkan feature map yang berbeda karena bobotnya berbeda. Kernel dengan bobot positif besar di satu sisi dan negatif di sisi lain berperan seperti detektor tepi pada arah tertentu, sedangkan kernel yang hampir seragam cenderung menghasilkan versi blur. Pada CNN, nilai kernel tidak diacak melainkan **dipelajari** melalui backpropagation sehingga filter menjadi selektif terhadap pola yang berguna untuk klasifikasi. Menjalankan ulang sel menghasilkan pola berbeda karena kernel di-generate ulang secara acak — inilah alasan CNN membutuhkan training, bukan kernel acak.

## Percobaan 4 — Filter Beauty dan Vintage

Kombinasi beberapa filter tradisional:
- **Beauty:** bilateral filter kuat (melembutkan tekstur) + penajaman ringan + kecerahan.
- **Vintage:** pergeseran warna ke arah hangat, penurunan kontras, noise film, dan *vignette*.

In [ ]:
def filter_beauty(citra):
    halus = cv2.bilateralFilter(citra, d=15, sigmaColor=90, sigmaSpace=90)
    tajam = cv2.filter2D(halus, -1, kernel_sharpen)
    return cv2.convertScaleAbs(tajam, alpha=1.05, beta=8)


def filter_vintage(citra):
    img = citra.astype(np.float32)
    r, g, b = cv2.split(img)
    r = np.clip(r * 1.12 + 15, 0, 255)
    g = np.clip(g * 0.98 + 8, 0, 255)
    b = np.clip(b * 0.85, 0, 255)
    img = cv2.merge([r, g, b])

    h, w = img.shape[:2]
    yy, xx = np.mgrid[0:h, 0:w]
    jarak = np.sqrt(((xx - w / 2) / (w / 2)) ** 2 + ((yy - h / 2) / (h / 2)) ** 2)
    vignette = np.clip(1.15 - 0.55 * jarak, 0, 1)[..., None]
    img = img * vignette

    noise = np.random.normal(0, 6, img.shape)
    return ke_uint8(img * 0.92 + noise + 8)


beauty = filter_beauty(img_rgb)
vintage = filter_vintage(img_rgb)

tampilkan([img_rgb, beauty, vintage], ["Asli", "Beauty", "Vintage"], ncols=3, skala=5.5)

## Percobaan 5 — Filter Anime / Cartoon

Kombinasi bilateral filter (warna diratakan) + deteksi tepi + operasi `bitwise_and` (garis tepi dipertahankan).

In [ ]:
def filter_kartun(image, metode_tepi="adaptive", d=9, sigma_warna=75,
                  sigma_ruang=75, blok=31, konstanta=18,
                  batas_bawah=30, batas_atas=90):
    """Efek kartun: bilateral filter + adaptive threshold / Canny."""
    warna = image.copy()
    for _ in range(2):
        warna = cv2.bilateralFilter(warna, d, sigma_warna, sigma_ruang)
    warna = cv2.medianBlur(warna, 5)

    abu = cv2.cvtColor(warna, cv2.COLOR_RGB2GRAY)
    abu = cv2.medianBlur(abu, 9)

    if metode_tepi == "adaptive":
        tepi = cv2.adaptiveThreshold(
            abu, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, blok, konstanta
        )
        kernel3 = np.ones((3, 3), np.uint8)
        tepi = cv2.morphologyEx(tepi, cv2.MORPH_OPEN, kernel3)
        tepi = cv2.morphologyEx(tepi, cv2.MORPH_CLOSE, kernel3)
    else:
        tepi = cv2.bitwise_not(cv2.Canny(abu, batas_bawah, batas_atas))

    tepi_rgb = cv2.cvtColor(tepi, cv2.COLOR_GRAY2RGB)
    return cv2.bitwise_and(warna, tepi_rgb)


kartun_adaptive = filter_kartun(img_rgb, metode_tepi="adaptive")
kartun_canny = filter_kartun(img_rgb, metode_tepi="canny")

tampilkan(
    [img_rgb, kartun_adaptive, kartun_canny],
    ["Asli", "Kartun (Adaptive Threshold)", "Kartun (Canny)"],
    ncols=3,
    skala=5.5,
)

## Percobaan 6 — Filter Malam

Efek malam: gamma dikurangi (lebih gelap), kanal biru dikuatkan, kanal merah diturunkan, ditambah *vignette*.

In [ ]:
def filter_malam(citra, gamma=1.45, intensitas_vignette=0.55):
    img = citra.astype(np.float32) / 255.0
    img = np.power(img, gamma)

    r, g, b = cv2.split(img)
    r = r * 0.82
    g = g * 0.95
    b = np.clip(b * 1.30, 0, 1)
    img = cv2.merge([r, g, b])

    h, w = img.shape[:2]
    yy, xx = np.mgrid[0:h, 0:w]
    jarak = np.sqrt(((xx - w / 2) / (w / 2)) ** 2 + ((yy - h / 2) / (h / 2)) ** 2)
    vignette = np.clip(1.10 - intensitas_vignette * jarak, 0, 1)[..., None]
    return ke_uint8(img * vignette * 255)


malam = filter_malam(img_rgb)
tampil_berdampingan(img_rgb, malam, "Asli", "Filter Malam")

## Percobaan 7 — Filter Pagi dan Pagi + Kabut

Filter pagi: citra lebih terang, hangat, dan kontrasnya sedikit dinaikkan. Kabut ditambahkan dengan mencampur warna putih memakai bobot yang lebih tebal di bagian atas (menyerupai kabut pagi).

In [ ]:
def filter_pagi(citra, gamma=0.88, kontras=1.08):
    img = citra.astype(np.float32) / 255.0
    img = np.power(img, gamma)

    r, g, b = cv2.split(img)
    r = r * 1.08
    g = g * 1.02
    b = b * 0.92
    img = cv2.merge([r, g, b])

    img = (img - 0.5) * kontras + 0.5 + 0.03
    return ke_uint8(np.clip(img, 0, 1) * 255)


def tambah_kabut(citra, intensitas=0.45):
    h = citra.shape[0]
    bobot = np.linspace(0.9, 0.15, h)[:, None, None]
    kabut = citra.astype(np.float32) * (1 - intensitas * bobot) + 255.0 * (intensitas * bobot)
    return ke_uint8(kabut)


pagi = filter_pagi(img_rgb)
pagi_kabut = tambah_kabut(pagi, intensitas=0.45)

tampilkan(
    [img_rgb, pagi, pagi_kabut],
    ["Asli", "Filter Pagi", "Filter Pagi + Kabut"],
    ncols=3,
    skala=5.5,
)

---

# Tugas Praktikum

## Tugas 1 — Mean Filter vs Median Filter pada Noise Salt & Pepper

### 1a. Menambahkan noise salt & pepper

In [ ]:
def salt_pepper(citra, rasio=0.02, seed=None):
    """Tambahkan noise salt (255) dan pepper (0) secara acak.

    Returns
    -------
    (citra_bernoise, posisi_piksel_noise)
    """
    if seed is not None:
        np.random.seed(seed)
    hasil = citra.copy()
    jumlah = int(rasio * citra.size)
    posisi = np.random.choice(citra.size, jumlah, replace=False)
    setengah = jumlah // 2
    hasil.flat[posisi[:setengah]] = 0
    hasil.flat[posisi[setengah:]] = 255
    return hasil, posisi


img_noise, posisi_noise = salt_pepper(img_gray, rasio=0.02, seed=42)
tampil_berdampingan(img_gray, img_noise, "Asli (grayscale)", "Salt & Pepper 2%")

### 1b. Mean filter 3x3 dan Median filter 3x3

Keduanya dibuat manual: mean filter memakai `konvolusi()` dengan kernel 1/9, median filter memakai *sliding window* lalu diambil nilai tengah.

In [ ]:
def median_filter(citra, ukuran=3):
    """Median filter manual dengan sliding window."""
    pad = ukuran // 2
    padded = np.pad(citra, pad, mode="edge")
    jendela = np.lib.stride_tricks.sliding_window_view(padded, (ukuran, ukuran))
    return np.median(jendela, axis=(-1, -2)).astype(np.uint8)


kernel_mean3 = np.ones((3, 3)) / 9
mean_3 = ke_uint8(konvolusi(img_noise, kernel_mean3, padding="replicate"))
median_3 = median_filter(img_noise, ukuran=3)


def mse(a, b):
    return float(np.mean((a.astype(np.float64) - b.astype(np.float64)) ** 2))


def psnr(a, b):
    nilai_mse = mse(a, b)
    return float("inf") if nilai_mse == 0 else 10 * math.log10(255 ** 2 / nilai_mse)


tampilkan(
    [img_noise, mean_3, median_3],
    ["Noisy (Salt & Pepper 2%)", "Mean Filter 3x3", "Median Filter 3x3"],
    ncols=3,
    skala=5.5,
)

# Posisi noise disimpan agar bisa dievaluasi secara terarah
mask_noise = np.zeros(img_noise.size, dtype=bool)
mask_noise[posisi_noise] = True
mask_noise = mask_noise.reshape(img_noise.shape)

print(f"{'Citra':<14}{'MAE pada piksel noise':>24}")
print("-" * 38)
for nama, hasil in [("Noisy", img_noise), ("Mean 3x3", mean_3), ("Median 3x3", median_3)]:
    mae_noise = np.abs(hasil[mask_noise].astype(float) - img_gray[mask_noise]).mean()
    print(f"{nama:<14}{mae_noise:>24.2f}")

### 1c. Uji terkontrol: papan uji sintetis

Foto hutan memiliki tekstur sangat rapat (daun dan rumput), sehingga MSE/PSNR terhadap citra asli juga dipengaruhi oleh hilangnya tekstur akibat smoothing — bukan hanya oleh noise. Agar pengaruh noise dan filter dapat diisolasi, pengujian metrik dilakukan pada **citra papan uji sintetis** (area datar + tepi tegas) dengan noise 5%.

Papan uji ini tetap citra keabuan, dan pola yang dipakai (kotak, lingkaran, latar datar) adalah standar untuk menguji filter penghilang noise.

In [ ]:
papan = np.full((240, 320), 90, dtype=np.uint8)
cv2.rectangle(papan, (40, 40), (160, 200), 200, -1)
cv2.circle(papan, (240, 120), 60, 150, -1)
papan = cv2.GaussianBlur(papan, (5, 5), 1.5)

papan_noise, _ = salt_pepper(papan, rasio=0.05, seed=7)
papan_mean = ke_uint8(konvolusi(papan_noise, kernel_mean3, padding="replicate"))
papan_median = median_filter(papan_noise, ukuran=3)

tampilkan(
    [papan, papan_noise, papan_mean, papan_median],
    ["Papan uji bersih", "Salt & Pepper 5%", "Mean 3x3", "Median 3x3"],
    ncols=2,
    skala=5,
)

print(f"{'Citra':<16}{'MSE':>10}{'PSNR (dB)':>14}")
print("-" * 40)
for nama, hasil in [("Noisy", papan_noise), ("Mean 3x3", papan_mean), ("Median 3x3", papan_median)]:
    print(f"{nama:<16}{mse(papan, hasil):>10.2f}{psnr(papan, hasil):>14.2f}")

### 1d. Perbesaran area noise pada foto (zoom)

Agar perbedaan terlihat jelas pada citra asli, berikut potongan area 120x160 piksel yang diperbesar.

In [ ]:
y0, y1, x0, x1 = 150, 270, 300, 460
potongan = [img_gray[y0:y1, x0:x1], img_noise[y0:y1, x0:x1],
            mean_3[y0:y1, x0:x1], median_3[y0:y1, x0:x1]]
tampilkan(
    potongan,
    ["Asli (crop)", "Noisy (crop)", "Mean 3x3 (crop)", "Median 3x3 (crop)"],
    ncols=2,
    skala=5,
)

### Analisis Tugas 1

- **Mean filter** mengganti setiap piksel dengan **rata-rata** 9 tetangganya. Piksel noise yang bernilai ekstrem (0 atau 255) tetap ikut dijumlahkan, sehingga nilainya "melebar" ke seluruh jendela. Akibatnya noise berkurang tetapi menyisakan bercak abu-abu dan citra menjadi **blur**.
- **Median filter** mengganti piksel dengan **nilai tengah** dari 9 nilai. Karena noise salt & pepper adalah outlier ekstrem, nilai tersebut hampir selalu terbuang saat diurutkan — asalkan jumlah noise dalam jendela kurang dari setengah. Hasilnya noise hilang **tanpa mengaburkan tepi**.
- Pada papan uji sintetis, Median 3x3 mencapai PSNR sekitar **51 dB** (MSE < 1), sedangkan Mean 3x3 hanya sekitar **28 dB** (MSE > 100). Selisih ini membuktikan median jauh lebih efektif menghapus noise impuls.
- Pada foto hutan, MAE pada piksel noise juga lebih kecil untuk median. Namun MSE/PSNR keseluruhan tidak dipakai sebagai patokan utama karena foto bertekstur rapat: mean maupun median sama-sama mengubah tekstur asli, sehingga metrik global menjadi bias.
- **Kesimpulan:** untuk noise impuls (salt & pepper), median filter jauh lebih efektif. Mean filter lebih cocok untuk noise Gaussian yang tersebar halus, bukan impuls.

## Tugas 2 — Pemilihan Ukuran Kernel Sharpening (3x3 vs 5x5)

### 2a. Membuat citra sedikit buram dan kedua kernel

In [ ]:
img_buram = cv2.GaussianBlur(img_gray, (9, 9), 2.0)

kernel_sharpen_3 = np.array([
    [0, -1, 0],
    [-1, 5, -1],
    [0, -1, 0],
], dtype=float)

kernel_sharpen_5 = np.array([
    [0, 0, -1, 0, 0],
    [0, -1, -2, -1, 0],
    [-1, -2, 17, -2, -1],
    [0, -1, -2, -1, 0],
    [0, 0, -1, 0, 0],
], dtype=float)

print("Jumlah kernel 3x3 :", kernel_sharpen_3.sum())
print("Jumlah kernel 5x5 :", kernel_sharpen_5.sum())

tajam_3 = ke_uint8(konvolusi(img_buram, kernel_sharpen_3, padding="replicate"))
tajam_5 = ke_uint8(konvolusi(img_buram, kernel_sharpen_5, padding="replicate"))

tampilkan(
    [img_buram, tajam_3, tajam_5],
    ["Input buram (Gaussian 9x9)", "Sharpen 3x3", "Sharpen 5x5"],
    ncols=3,
    skala=5.5,
)

### 2b. Evaluasi: kejelasan tepi vs noise

Metrik yang dipakai:
- **Variance of Laplacian** — semakin tinggi, semakin banyak detail/tepi tajam.
- **Kekuatan tepi** — rata-rata magnitudo gradien (Sobel).
- **Kadar noise** — simpangan baku selisih citra dengan versi median-blur 5x5 (komponen frekuensi tinggi yang tidak terstruktur).

In [ ]:
def ketajaman(citra):
    return float(cv2.Laplacian(citra, cv2.CV_64F).var())


def kekuatan_tepi(citra):
    gx = cv2.Sobel(citra, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(citra, cv2.CV_64F, 0, 1, ksize=3)
    return float(np.mean(np.hypot(gx, gy)))


def kadar_noise(citra):
    halus = cv2.medianBlur(citra, 5)
    return float(np.std(citra.astype(np.float64) - halus.astype(np.float64)))


print(f"{'Citra':<20}{'Var Laplacian':>15}{'Kekuatan tepi':>16}{'Kadar noise':>14}")
print("-" * 65)
for nama, hasil in [("Input buram", img_buram), ("Sharpen 3x3", tajam_3), ("Sharpen 5x5", tajam_5)]:
    print(f"{nama:<20}{ketajaman(hasil):>15.2f}{kekuatan_tepi(hasil):>16.2f}{kadar_noise(hasil):>14.2f}")

In [ ]:
metrik = {
    "Var Laplacian": [ketajaman(img_buram), ketajaman(tajam_3), ketajaman(tajam_5)],
    "Kekuatan tepi": [kekuatan_tepi(img_buram), kekuatan_tepi(tajam_3), kekuatan_tepi(tajam_5)],
    "Kadar noise": [kadar_noise(img_buram), kadar_noise(tajam_3), kadar_noise(tajam_5)],
}
label = ["Input buram", "Sharpen 3x3", "Sharpen 5x5"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (judul, nilai) in zip(axes, metrik.items()):
    ax.bar(label, nilai, color=["#9e9e9e", "#42a5f5", "#ef5350"])
    ax.set_title(judul)
    ax.tick_params(axis="x", rotation=15)
    for i, v in enumerate(nilai):
        ax.text(i, v, f"{v:.1f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

### Analisis Tugas 2

| Aspek | Sharpen 3x3 | Sharpen 5x5 |
|---|---|---|
| Kejelasan tepi | Tepi jelas, halus, tidak berlebihan | Tepi lebih tegas dan kontras |
| Noise / distorsi | Rendah | Lebih tinggi (noise ikut diperkuat) |
| Artefak | Minimal | Muncul *halo*/garis terang di sekitar tepi |
| Biaya komputasi | Ringan | Lebih berat (25 piksel per output) |

Kernel 5x5 memiliki jangkauan lebih luas sehingga mampu mengembalikan detail yang hilang akibat blur yang lebar, tetapi karena jumlah koefisien negatifnya lebih banyak, noise dan tepi palsu juga ikut diperkuat.

**Kesimpulan:** untuk pemakaian sehari-hari, **kernel 3x3 lebih optimal**. Hasilnya sudah cukup tajam, noise tetap terkendali, dan tidak menimbulkan halo. Kernel 5x5 hanya dipakai bila citra memang sangat buram dan noise-nya rendah (misalnya hasil downscale atau motion blur ringan), atau setelah proses denoising.

## Tugas 3 — Filter Kartun Sederhana

Fungsi `kartun(image)` menggabungkan:
1. **Bilateral filter** (2 iterasi) untuk meratakan warna tanpa menghilangkan tepi.
2. **Adaptive thresholding** (default) atau **Canny** untuk garis tepi.
3. **Operasi logika `bitwise_and`** untuk menggabungkan warna rata dengan garis tepi.

In [ ]:
def posterize(citra, level=12):
    """Kuantisasi warna agar area warna terlihat rata seperti ilustrasi."""
    langkah = max(1, 256 // level)
    return np.clip((citra // langkah) * langkah + langkah // 2, 0, 255).astype(np.uint8)


def kartun(image, metode_tepi="adaptive", d=9, sigma_warna=75, sigma_ruang=75,
           blok=31, konstanta=18, batas_bawah=30, batas_atas=90,
           level_warna=0, ukuran_median=9, ukuran_morfologi=3):
    """Filter kartun sederhana dari kombinasi filter tradisional.

    Parameters
    ----------
    image            : citra RGB (uint8)
    metode_tepi      : 'adaptive' (adaptive threshold) atau 'canny'
    d, sigma_warna, sigma_ruang : parameter bilateral filter
    blok, konstanta  : parameter adaptive threshold (blok harus bilangan ganjil)
    batas_bawah, batas_atas : parameter Canny
    level_warna      : 0 = tanpa posterize, >0 = jumlah level warna
    ukuran_median    : ukuran median blur sebelum deteksi tepi
    ukuran_morfologi : ukuran kernel opening/closing untuk membersihkan garis

    Returns
    -------
    citra RGB hasil efek kartun
    """
    if image.ndim != 3:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)

    warna = image.copy()
    for _ in range(2):
        warna = cv2.bilateralFilter(warna, d, sigma_warna, sigma_ruang)
    warna = cv2.medianBlur(warna, 5)

    if level_warna > 0:
        warna = posterize(warna, level_warna)

    abu = cv2.cvtColor(warna, cv2.COLOR_RGB2GRAY)
    abu = cv2.medianBlur(abu, ukuran_median)

    if metode_tepi == "adaptive":
        tepi = cv2.adaptiveThreshold(
            abu, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, blok, konstanta
        )
        kernel = np.ones((ukuran_morfologi, ukuran_morfologi), np.uint8)
        tepi = cv2.morphologyEx(tepi, cv2.MORPH_OPEN, kernel)
        tepi = cv2.morphologyEx(tepi, cv2.MORPH_CLOSE, kernel)
    elif metode_tepi == "canny":
        tepi = cv2.bitwise_not(cv2.Canny(abu, batas_bawah, batas_atas))
    else:
        raise ValueError("metode_tepi harus 'adaptive' atau 'canny'")

    tepi_rgb = cv2.cvtColor(tepi, cv2.COLOR_GRAY2RGB)
    return cv2.bitwise_and(warna, tepi_rgb)


hasil_kartun = kartun(img_rgb)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
axes[0].imshow(img_rgb)
axes[0].set_title("Citra Asli")
axes[0].axis("off")
axes[1].imshow(hasil_kartun)
axes[1].set_title("Citra Efek Kartun")
axes[1].axis("off")
plt.tight_layout()
plt.show()

### Variasi parameter (opsional)

Eksperimen kecil untuk melihat pengaruh ukuran bilateral dan ketebalan garis tepi.

In [ ]:
variasi = {
    "Adaptive (default)": kartun(img_rgb),
    "Adaptive + posterize 12": kartun(img_rgb, level_warna=12),
    "Canny (30, 90)": kartun(img_rgb, metode_tepi="canny"),
    "Bilateral besar (d=15)": kartun(img_rgb, d=15, sigma_warna=120, sigma_ruang=120),
}
tampilkan(
    [img_rgb] + list(variasi.values()),
    ["Asli"] + list(variasi.keys()),
    ncols=2,
    skala=6,
)

### Analisis Tugas 3

- **Bilateral filter** membuat warna di dalam satu area menjadi rata (seperti cat solid) tetapi batas antarobjek tetap tegas. Semakin besar `d`, `sigma_warna`, dan `sigma_ruang`, semakin "flat" hasilnya tetapi prosesnya semakin lambat.
- **Adaptive threshold** memakai ambang lokal sehingga garis tepi tetap muncul pada area terang maupun gelap. Nilai `blok` menentukan ukuran lingkungan dan `konstanta` menentukan seberapa mudah suatu piksel dianggap tepi. Pada foto bertekstur rapat (daun, rumput), `blok` kecil dan `konstanta` rendah membuat garis pecah menjadi bintik hitam; karena itu dipakai `blok = 31`, `konstanta = 18`, dan hasil ambang dibersihkan dengan *opening* + *closing* 3x3. **Canny** menghasilkan garis lebih tipis dan bersih, tetapi ambangnya harus disesuaikan dengan pencahayaan citra.
- **`bitwise_and`** mempertahankan warna hasil bilateral hanya pada piksel yang bukan garis tepi (garis bernilai 0). Hasilnya tampak seperti ilustrasi kartun dengan garis hitam.

---

# Kesimpulan Akhir

1. **Konvolusi** adalah inti dari filter spasial: setiap piksel baru dihitung dari kombinasi linear piksel tetangganya sesuai bobot kernel. Implementasi manual (loop maupun vectorized) menghasilkan nilai yang sama dengan library.
2. **Low pass filter** (average, Gaussian) menekan frekuensi tinggi → citra halus dan noise berkurang, tetapi tepi ikut kabur.
3. **High pass filter** (high pass, sharpen, Laplacian, emboss) menonjolkan frekuensi tinggi → tepi dan detail menjadi jelas.
4. **Pemilihan kernel** menentukan efek: simetris untuk smoothing, asimetris untuk emboss/deteksi arah, dan jumlah koefisien menentukan kecerahan hasil.
5. **Median filter** unggul untuk noise impulsif (salt & pepper) karena tidak terpengaruh outlier, sedangkan **mean filter** justru menyebarkan noise tersebut.
6. **Sharpening 3x3** lebih seimbang untuk pemakaian umum; **5x5** lebih tajam tetapi memperkuat noise dan memunculkan halo.
7. **Filter modern** (bilateral, guided) mampu menghaluskan sambil mempertahankan tepi, dan dapat dikombinasikan dengan filter tradisional untuk efek seperti beauty, vintage, kartun, malam, dan pagi.

---

## Checklist Pengumpulan

- [ ] Nama dan NIM sudah diisi pada header.
- [ ] File `djawatan.jpg` berhasil diupload dan citra tampil.
- [ ] Seluruh sel berjalan tanpa error (`Runtime > Restart and Run All`).
- [ ] Notebook disimpan sebagai **`Week7.ipynb`**.
- [ ] Notebook diunggah ke **GitHub** (repository praktikum).
- [ ] Tautan GitHub dikumpulkan sesuai instruksi kelas.